# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### The Rule
* **Population:** All statistically reliable pages (pages with >= 500 impressions). We do NOT hard-filter by impression tiers, ensuring we don't blind the rule to 69% of our dataset.
* **Evidence:** We compare the page's actual Click-Through Rate (CTR) against the baseline average CTR for its specific `position_tier`.
* **Condition:** If the page's CTR is severely underperforming its peers (e.g., it is less than half of its tier's average CTR)...
* **Action:** Flag the page for "Metadata Review". To prioritize, we rank the queue by the **total missed clicks** (the CTR gap multiplied by impression volume). This naturally pushes high-volume pages to the top without artificially making moderate-volume pages invisible.

### Reason Codes
* `SEVERE_CTR_GAP`: Page's CTR is significantly worse than expected for its ranking position.

In [2]:
import duckdb
import os, getpass
import numpy as np
import pandas as pd

con = duckdb.connect()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token: ')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT_DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"

# 1. Pull the data just like we did in w03
feature_query = f"""
SELECT
    content_hash_id,
    client_hash_id,
    SUM(gsc_impressions)        AS total_impressions,
    SUM(gsc_clicks)             AS total_clicks,
    AVG(gsc_avg_position)       AS average_position,
    SUM(ga4_sessions)           AS total_sessions,
    SUM(ga4_engaged_sessions)   AS total_engaged_sessions,
    SUM(ga4_pageviews)          AS total_pageviews
FROM {FACT_DAILY}
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
  AND client_has_ga4 IS TRUE
  AND client_has_gsc IS TRUE
GROUP BY content_hash_id, client_hash_id
"""
monthly_features = con.sql(feature_query).df()

# 2. Add the CTR column safely
monthly_features['ctr'] = np.where(
    monthly_features['total_impressions'] > 0,
    monthly_features['total_clicks'] / monthly_features['total_impressions'],
    np.nan
)

# 3. Apply your 500-impression exclusion rule
df = monthly_features[monthly_features['total_impressions'] >= 500].copy()

print(f"Data loaded! We have {len(df):,} reliable pages to work with.")


Data loaded! We have 42,174 reliable pages to work with.


In [3]:
import pandas as pd
import numpy as np

bins = [-float("inf"), 3, 10, 20, 50, float("inf")]
labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]

# 1. Bucket the numeric values
df["position_tier"] = pd.cut(df["average_position"], bins=bins, labels=labels)

# 2. Add "no_data" to the allowed categories, then fill the NaNs
df["position_tier"] = df["position_tier"].cat.add_categories(["no_data"]).fillna("no_data")


In [4]:
test_positions = pd.Series([3.0, 3.1, 10.0, 10.1, 50.0, 50.1])
test_bins = pd.cut(test_positions, bins=bins, labels=labels)
print(list(zip(test_positions, test_bins)))

[(3.0, 'top_3'), (3.1, 'page_1'), (10.0, 'page_1'), (10.1, 'striking'), (50.0, 'page_3_5'), (50.1, 'deep')]


In [5]:
ctr_by_pos = df.groupby("position_tier")["ctr"].agg(["mean", "count"])
print(ctr_by_pos)


                   mean  count
position_tier                 
top_3          0.003944   4050
page_1         0.003528  19461
striking       0.002944   8939
page_3_5       0.001466   9354
deep           0.000305    370


In [6]:
# Convert fraction to a percentage (0.003944 -> 0.3944)
df["ctr_pct"] = df["ctr"] * 100

# Group by the tier and measure our new percentage column
ctr_by_pos = df.groupby("position_tier")["ctr_pct"].agg(["mean", "count"])

# Round the mean to 2 decimal places so it's super clean to read
print(ctr_by_pos.round({"mean": 2}))


               mean  count
position_tier             
top_3          0.39   4050
page_1         0.35  19461
striking       0.29   8939
page_3_5       0.15   9354
deep           0.03    370


Signal A — CTR vs. position_tier. Verdict: CONFIRMED.
CTR declines monotonically across all five tiers with zero reversals: top_3 (0.39%, n=4,050) → page_1 (0.35%, n=19,461) → striking (0.29%, n=8,939) → page_3_5 (0.15%, n=9,354) → deep (0.03%, n=370). Even the smallest bucket (deep) has 370 pages, enough to trust the mean. This confirms that click behavior tracks search position, which justifies comparing a page's CTR against its own position-tier average — rather than a single global threshold — when flagging underperformance. This mirrors FlyRank's real CTR-fix logic.

In [7]:
# We start at 299 to respect the official "moderate >= 300" rule,
# even though our actual data doesn't start until 500.
imp_bins = [299, 2999, 29999, float("inf")]
imp_labels = ["moderate", "good", "excellent"]

# Bucket it! No need to fillna("no_data") because we don't have that edge case anymore.
df["impression_tier"] = pd.cut(df["total_impressions"], bins=imp_bins, labels=imp_labels)

# Check the bucket table
imp_by_tier = df.groupby("impression_tier")["ctr_pct"].agg(["mean", "count"])
print(imp_by_tier.round({"mean": 2}))


                 mean  count
impression_tier             
moderate         0.29  29085
good             0.30  12603
excellent        0.25    486


Signal B — Impressions vs. CTR. Verdict: CONFIRMED (independence).
CTR stays roughly flat across impression-volume tiers: moderate (0.29%, n=29,085) → good (0.30%, n=12,603) → excellent (0.25%, n=486). Unlike Signal A's monotonic 10x decline across position tiers, there's no meaningful trend here — the small movement (0.29→0.30→0.25) doesn't consistently increase or decrease with volume, and even the smallest bucket (excellent, n=486) is well above the ~30-sample threshold for a stable mean. This confirms that impression volume and CTR are largely independent signals — a low-CTR page doesn't inherently have different click behavior at high volume vs. low volume. But it changes the stakes: fixing the same CTR gap on an excellent-tier page (thousands of impressions) recovers far more absolute clicks than fixing it on a moderate-tier page. This justifies using impression volume as a multiplier on urgency/priority, not as a signal of CTR quality itself.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# (Steps 1-3 from before)
raw_ctr_by_pos = df.groupby("position_tier")["ctr"].mean()
df["tier_avg_ctr"] = df["position_tier"].map(raw_ctr_by_pos)
df["ctr_gap"] = (df["tier_avg_ctr"] - df["ctr"]).clip(lower=0)
df["missed_clicks"] = df["ctr_gap"] * df["total_impressions"]

# Step 4: The Bouncer (The Condition Flag)
# True if the page's actual CTR is less than half of what its peers get
df["is_flagged"] = df["ctr"] < (df["tier_avg_ctr"] / 2)

# Step 5: Filter the dataset to ONLY flagged pages
flagged_queue = df[df["is_flagged"] == True].copy()

# Step 6: Rank the broken pages by how many clicks we lost (highest first)
ranked_queue = flagged_queue.sort_values("missed_clicks", ascending=False)

# Add the explicit reason code and action label for the CSV!
ranked_queue["reason_code"] = "SEVERE_CTR_GAP"
ranked_queue["action"] = "metadata_review"

import os
os.makedirs("../outputs", exist_ok=True)

# Check the true top 5!
print(f"Total flagged pages: {len(ranked_queue)}")
print("\nTop 5 Highest Priority Pages:")
print(ranked_queue[["content_hash_id", "position_tier", "total_impressions", "ctr", "tier_avg_ctr", "missed_clicks", "reason_code", "action"]].head(5))

# Finally, write the CSV for the assignment requirement
ranked_queue.to_csv("../outputs/baseline_action_score.csv", index=False)


Total flagged pages: 18212

Top 5 Highest Priority Pages:
                 content_hash_id position_tier  total_impressions       ctr  \
227499  content_44f34c0a90047651        page_1           212404.0  0.000113   
188136  content_8d7d99f109e19aa2         top_3           203497.0  0.001420   
60268   content_fec55986a1868d62         top_3           118018.0  0.000000   
39479   content_cd3d932d4e1c8db0        page_1            89332.0  0.000045   
128244  content_046fc480045b88f5        page_1            83788.0  0.000072   

        tier_avg_ctr  missed_clicks     reason_code           action  
227499      0.003528     725.432795  SEVERE_CTR_GAP  metadata_review  
188136      0.003944     513.528236  SEVERE_CTR_GAP  metadata_review  
60268       0.003944     465.425914  SEVERE_CTR_GAP  metadata_review  
39479       0.003528     311.193360  SEVERE_CTR_GAP  metadata_review  
128244      0.003528     289.632262  SEVERE_CTR_GAP  metadata_review  



At a 50%-below-tier-average threshold, this rule flags 18,212 pages (43% of reliable pages) — too many for weekly human review capacity. This isn't evidence that rule-based flagging is inherently broken; a stricter threshold (e.g., CTR < 30% of tier average) would shrink the list, but picking that number by hand is exactly the kind of unmaintainable, un-principled tuning Harris warned about — there's no data-driven way to know if 50% or 30% or 25% is 'correct' for every position tier and client. That's the specific gap a learned model can close: finding the right cutoff per segment from data, instead of us guessing one global number.

The threshold (50% vs 25% of tier average) doesn't change the top-20 — these are extreme outliers that clear either bar. But it does change the size of the full flagged queue (18,212 pages), which still matters if this queue is ever consumed beyond just 'top 20 this week.' The real fix isn't picking a better global threshold — it's that a single global cutoff can't distinguish 'this week's top 20' from 'the reviewable backlog' at all. That's a capacity/pagination problem this baseline doesn't solve, and arguably a place a model-based ranking (rather than a binary threshold) naturally does better, since it can produce a continuous priority score instead of a yes/no flag

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

> **Design Limitation Note:** Currently, our rule assigns the global reason code `SEVERE_CTR_GAP` to every flagged page. However, our manual review reveals a gap between "the rule fired" and "the rule explains itself well." A natural next step for an ML model would be to split this into more granular codes (e.g., `SEVERE_CTR_GAP_SUSPECTED_ZERO_CLICK` vs `SEVERE_CTR_GAP_LEGITIMATE`) since our baseline rule cannot distinguish them automatically. For this exercise, all pages below carry the `SEVERE_CTR_GAP` reason code.

**Rank 1 (content_44f34c0a... - page_1):**
* **Action:** Flagged for Metadata Review (#1 in queue, highest missed clicks).
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 212,404 impressions on Page 1 but a microscopic CTR of 0.011% (vs 0.35% average), bleeding ~725 clicks.
* **What would make this wrong:** **Zero-Click SERP**. High volume and near-zero CTR strongly suggests Google answers the query directly on the SERP.
* **Verdict:** True Positive. Stakes are too high (212k imp) to dismiss without a 5-minute manual check.

**Rank 2 (content_8d7d99f1... - top_3):**
* **Action:** Flagged for Metadata Review (#2 in queue).
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 203,497 impressions in `top_3`, with 0.14% CTR (vs 0.39% avg), missing ~513 clicks.
* **What would make this wrong:** **Branded Search**. Users are searching for a competitor and skipping our result. 
* **Verdict:** True Positive. Getting some clicks (289), so not a zero-click SERP. Highly likely a **Legitimately broken title/snippet**.

**Rank 3 (content_fec55986... - top_3):**
* **Action:** Flagged for Metadata Review (#3 in queue).
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 118,018 impressions in `top_3`, but literally 0 clicks.
* **What would make this wrong:** **Image Carousel / Instant Answer**. Getting 0 accidental clicks on 118k top-3 impressions implies it's unclickable.
* **Verdict:** False Positive (Highly Likely). The 0.00% CTR strongly points to a non-standard SERP feature.

**Rank 4 (content_cd3d932d... - page_1):**
* **Action:** Flagged for Metadata Review.
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 89,332 impressions with 0.004% CTR.
* **What would make this wrong:** **Zero-Click SERP**. Like Rank 1, massive impressions but only 4 clicks.
* **Verdict:** False Positive (Likely).

**Rank 5 (content_046fc480... - page_1):**
* **Action:** Flagged for Metadata Review.
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 83,788 impressions with 0.007% CTR.
* **What would make this wrong:** **Seasonality**. Could be an out-of-date title (e.g. "2025") for a suddenly popular search term.
* **Verdict:** True Positive. Easy fix if it's just a seasonal mismatch.

**Rank 6 (content_306bc78d... - top_3):**
* **Action:** Flagged for Metadata Review.
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 80,821 impressions, 0.04% CTR (vs 0.39%).
* **What would make this wrong:** **Legitimately broken title**. Similar to Rank 2, it gets 35 clicks, meaning it's clickable but being heavily ignored.
* **Verdict:** True Positive. Needs a better meta description.

**Rank 7 (content_9540d884... - page_1):**
* **Action:** Flagged for Metadata Review.
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 79,292 impressions, 0.01% CTR.
* **What would make this wrong:** **Seasonality** or **Small-tier artifact**. If it fluctuated near the bottom of page 1, the average might be too strict.
* **Verdict:** True Positive. 

**Rank 8 (content_9ef3d751... - top_3):**
* **Action:** Flagged for Metadata Review.
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 89,229 impressions, 0.10% CTR (vs 0.39%).
* **What would make this wrong:** **Branded Search**. Gets 92 clicks, suggesting heavy competitor dominance.
* **Verdict:** True Positive.

**Rank 9 (content_8e1334d6... - page_1):**
* **Action:** Flagged for Metadata Review.
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 71,863 impressions, 0.00% CTR (0 clicks).
* **What would make this wrong:** **Zero-Click SERP / Image Carousel**. 0 clicks on 71k impressions is statistically impossible for standard text.
* **Verdict:** False Positive (Highly Likely).

**Rank 16 (content_f6723f02... - striking):**
* **Action:** Flagged for Metadata Review.
* **Why it's here:** (Reason: SEVERE_CTR_GAP) 69,822 impressions, 0.018% CTR (vs 0.29% tier average).
* **What would make this wrong:** **Seasonality**. Spiked in impressions but underperformed due to mismatch in seasonal intent.
* **Verdict:** True Positive.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks
Our prime "looks wrong" candidate is **Rank 3** (`content_fec55986...`). It generated 118,018 impressions in the `top_3` tier but received absolutely zero clicks. In standard search results, it is almost mathematically impossible to get 118k impressions in the top 3 spots without getting even a single accidental click. This heavily implies our rule got tricked by a non-standard SERP feature (like an Image Carousel or a pure instant-answer snippet) where the result isn't actually a clickable text link. This is a clear False Positive.

### Leakage Check
I have verified all features used in our scoring (`total_impressions`, `total_clicks`, `ctr`, `average_position`, `total_engaged_sessions`). 
None of these features rely on FlyRank's internal product flags (like a proprietary `health_score`), and because our SQL query strictly bounded the `report_date` to the exact month of March, no future data windows leaked into our decision point.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [10]:
# Filter down to your eligible pages first
eligible_df = df[df['total_impressions'] >= 500]

# 1. Look at the distribution of clicks to see what's normal
print("--- Click Distribution ---")
print(eligible_df['total_clicks'].describe(percentiles=[0.25, 0.5, 0.75, 0.90]))

# 2. Test your gut instinct (20 clicks)
surviving_20 = (eligible_df['total_clicks'] >= 20).sum()
total_eligible = len(eligible_df)
print(f"\nPages with 20+ clicks: {surviving_20} out of {total_eligible}")


--- Click Distribution ---
count    42174.000000
mean        11.000308
std         41.776566
min          0.000000
25%          1.000000
50%          3.000000
75%         10.000000
90%         25.000000
max       5668.000000
Name: total_clicks, dtype: float64

Pages with 20+ clicks: 5587 out of 42174


In [11]:
# Filter down to your eligible pages first
eligible_df = df[df['total_impressions'] >= 500]

# 1. Look at the distribution of clicks to see what's normal
print("--- Click Distribution ---")
print(eligible_df['total_clicks'].describe(percentiles=[0.25, 0.5, 0.75, 0.90]))

# 2. Test your gut instinct (20 clicks)
surviving_20 = (eligible_df['total_clicks'] >= 10).sum()
total_eligible = len(eligible_df)
print(f"\nPages with 20+ clicks: {surviving_20} out of {total_eligible}")


--- Click Distribution ---
count    42174.000000
mean        11.000308
std         41.776566
min          0.000000
25%          1.000000
50%          3.000000
75%         10.000000
90%         25.000000
max       5668.000000
Name: total_clicks, dtype: float64

Pages with 20+ clicks: 10757 out of 42174


In [12]:
# Only look at pages that actually got clicks
clicked_df = eligible_df[eligible_df['total_clicks'] > 0].copy()

# Calculate the engagement rate
clicked_df['engagement_rate'] = clicked_df['total_engaged_sessions'] / clicked_df['total_clicks']

# 1. Look at the distribution to find the bottom tail
print("--- Engagement Rate Distribution ---")
print(clicked_df['engagement_rate'].describe(percentiles=[0.10, 0.25, 0.5, 0.75, 0.90]))

# 2. Test a hypothetical cutoff (e.g., the bottom 20% or 10% engagement)
# We can adjust this 0.20 based on what the distribution tells us
test_threshold = 0.20 
low_engagement_count = (clicked_df['engagement_rate'] < test_threshold).sum()
total_clicked = len(clicked_df)

print(f"\nPages with <{test_threshold*100}% engagement rate: {low_engagement_count} out of {total_clicked}")


--- Engagement Rate Distribution ---
count    34375.000000
mean         0.068477
std          0.191052
min          0.000000
10%          0.000000
25%          0.000000
50%          0.000000
75%          0.055815
90%          0.200000
max          5.000000
Name: engagement_rate, dtype: float64

Pages with <20.0% engagement rate: 30720 out of 34375


In [13]:
clicked_df[clicked_df['total_clicks'] >= 50]['engagement_rate'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

count    1725.000000
mean        0.051103
std         0.047994
min         0.000000
10%         0.000000
25%         0.013889
50%         0.038462
75%         0.076923
90%         0.115385
max         0.413793
Name: engagement_rate, dtype: float64